# Assignment 3 — Multi-Agent Collaboration

Goal: build three specialized agents that collaborate on a single query,
coordinated by a simple sequential controller:

1. **SearchAgent** — retrieves factual information from the web
2. **AnalysisAgent** — reasons over the facts and performs any needed
   calculations
3. **ReportAgent** — turns the facts + analysis into a clean,
   human-readable summary

This notebook is self-contained.


## 0. Setup

In [1]:
!pip install -q transformers accelerate ddgs

In [2]:
import re
import ast
import operator as op

from transformers import pipeline
from ddgs import DDGS


## 1. Model and tools (shared across all three agents)

In [3]:
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # <- lighter alternative for CPU/local testing

generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    max_new_tokens=300,
    do_sample=False,
    temperature=0.0,
)


def llm(prompt: str, max_new_tokens: int = 300) -> str:
    out = generator(prompt, max_new_tokens=max_new_tokens, return_full_text=False)
    return out[0]["generated_text"].strip()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
def search_tool(query: str, max_results: int = 4) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No results found."
        return "\n".join(f"- {r['title']}: {r['body']}" for r in results)
    except Exception as e:
        return f"Search error: {e}"


_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg, ast.UAdd: op.pos,
    ast.FloorDiv: op.floordiv,
    # Note: '%' is reserved for percent notation (see _preprocess_percent below),
    # not modulo - that avoids ambiguity with expressions like '10%*69081996'.
}

PERCENT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*%")


def _preprocess_percent(expression: str) -> str:
    """Turn 'N%' into '(N/100)' so the model can write percentages naturally,
    e.g. '10%*69081996' -> '(10/100)*69081996'."""
    return PERCENT_RE.sub(r"(\1/100)", expression)


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression node: {ast.dump(node)}")


def calculator_tool(expression: str) -> str:
    try:
        expression = _preprocess_percent(expression)
        tree = ast.parse(expression, mode="eval")
        return str(_eval_node(tree.body))
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


# Looks for a standalone arithmetic expression the analysis agent decided to compute,
# e.g. "CALC: 67000000 * 0.10"
CALC_RE = re.compile(r"CALC:\s*(.+)")


## 2. Agent definitions

Each agent is a small class with one job, its own prompt template, and (for
Search/Analysis) a tool it can call.


In [5]:
class SearchAgent:
    """Retrieves and condenses factual information relevant to the question."""

    PROMPT = """You are SearchAgent. Read the raw search results below and extract only
the facts relevant to the question, as a short bullet list. Do not answer
the question yet - just extract facts.

Question: {question}

Raw search results:
{raw_results}

Relevant facts (bullet list):"""

    def run(self, question: str) -> str:
        raw_results = search_tool(question)
        prompt = self.PROMPT.format(question=question, raw_results=raw_results)
        facts = llm(prompt)
        return facts


class AnalysisAgent:
    """Reasons over the facts and performs any calculation needed."""

    PROMPT = """You are AnalysisAgent. Given the question and the facts below, do the
reasoning needed to answer it. If a calculation is required, output a line
in the exact form:
CALC: <arithmetic expression>
and nothing else on that line. Otherwise, just write your analysis directly.

Question: {question}

Facts:
{facts}

Analysis:"""

    def run(self, question: str, facts: str) -> str:
        prompt = self.PROMPT.format(question=question, facts=facts)
        analysis = llm(prompt)

        match = CALC_RE.search(analysis)
        if match:
            expression = match.group(1).strip()
            result = calculator_tool(expression)
            analysis += f"\n(calculator[{expression}] = {result})"
        return analysis


class ReportAgent:
    """Synthesizes facts + analysis into a clear, human-readable report."""

    PROMPT = """You are ReportAgent. Write a clear, concise answer to the question for a
non-technical reader, using the facts and analysis below. 2-4 sentences.
Do not mention "Facts" or "Analysis" as section labels - just answer
naturally.

Question: {question}

Facts:
{facts}

Analysis:
{analysis}

Final report:"""

    def run(self, question: str, facts: str, analysis: str) -> str:
        prompt = self.PROMPT.format(question=question, facts=facts, analysis=analysis)
        return llm(prompt)


## 3. Controller — sequential orchestration

The controller calls the three agents in a fixed order: Search → Analysis →
Report, passing each agent's output forward as the next agent's input.


In [6]:
class Controller:
    def __init__(self):
        self.search_agent = SearchAgent()
        self.analysis_agent = AnalysisAgent()
        self.report_agent = ReportAgent()

    def run(self, question: str, verbose: bool = True) -> str:
        if verbose:
            print("=== SearchAgent ===")
        facts = self.search_agent.run(question)
        if verbose:
            print(facts, "\n")

        if verbose:
            print("=== AnalysisAgent ===")
        analysis = self.analysis_agent.run(question, facts)
        if verbose:
            print(analysis, "\n")

        if verbose:
            print("=== ReportAgent ===")
        report = self.report_agent.run(question, facts, analysis)
        if verbose:
            print(report)

        return report


## 4. Example run

In [7]:
controller = Controller()

question = "What is the current population of France, and what would 5% of that be?"
final_report = controller.run(question)

print("\n=== FINAL REPORT ===")
print(final_report)


=== SearchAgent ===


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning

- Population of France as of 1 January 2026: 66,792,845
- Total population of France including overseas territories: 69,081,996
- Median age in France as of 2025: 41.74 years
- Male population in France as of 2025: 32.36 million
- Female population in France as of 2025: 34.39 million

These facts provide information about the current population of France and its demographic composition based on available data up to 2025. However, please note that these figures may have changed since then due to various factors such as births, deaths, migration, etc., so it's important to verify them against more recent sources if needed. To directly answer your question "What is the current population of France, and what would 5% of that be? ", we need to calculate:

Current population of France = 66,792,845
5% of this population = 66,792,845 * 0.05 = 3,339,642.25 ≈ 3,339,642 (rounded down) 

=== AnalysisAgent ===


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The current population of France is approximately 66,792,845 people. Five percent of this population would be around 3,339,642 people. This number can vary slightly depending on how you round the result, but for most practical purposes, rounding down gives us an accurate estimate. 

Calculation: 66,792,845 * 0.05 = 3,339,642.25 ≈ 3,339,642 

=== ReportAgent ===
The current population of France is approximately 66,792,845 people, which translates to roughly 3,339,642 individuals when calculating five percent of the total population. This figure reflects the median age of 41.74 years and includes both male and female populations. For any specific needs or further calculations, please refer to the latest official statistics from reliable sources.

=== FINAL REPORT ===
The current population of France is approximately 66,792,845 people, which translates to roughly 3,339,642 individuals when calculating five percent of the total population. This figure reflects the median age of 41.74 years

In [8]:
# A second example with no calculation required, to show AnalysisAgent
# gracefully skipping the calculator step.
question2 = "Who is the current Secretary-General of the United Nations?"
final_report2 = controller.run(question2)

print("\n=== FINAL REPORT ===")
print(final_report2)


=== SearchAgent ===


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- António Guterres is the current Secretary-General of the United Nations.
- He was elected in October 2016 and became the first European to hold the position since Kurt Waldheim in 1981. To be continued... 

=== AnalysisAgent ===


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The current Secretary-General of the United Nations is António Guterres. This can be determined from the fact that he has been serving as the Secretary-General since October 2016, making him the person currently holding this position. No further information or calculations were necessary for this straightforward identification based on the given data. There's no need to continue with "to be continued" as there's already enough context provided about his election date and the historical significance of being the first European to hold the position since Kurt Waldheim in 1981. Therefore, I don't need to perform any arithmetic expressions or provide additional details beyond stating who the current Secretary-General is. 

No specific reasoning or calculations were involved; only direct information from the given facts was used to make the determination. Thus, the final answer is:

António Guterres. 

=== ReportAgent ===
The current Secretary-General of the United Nations is António Guterr

## Notes / extensions

- This controller is **sequential and rule-based** by design (Search ->
  Analysis -> Report, always in that order), matching the assignment's
  "simple controller" requirement. Assignment 4 replaces this with a
  graph-based orchestrator that can branch conditionally.
- Each agent only knows about its own prompt and tool; the Controller is the
  only component that knows the overall pipeline order, which keeps the
  agents independently testable.
